In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc

def run_fraud_risk_engine():
    print("🚀 Ingesting heavy-duty transactional financial data...")

    # Locate the file dynamically in the workspace
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError("❌ No CSV file found. Please upload your Kaggle dataset into the Colab side-panel!")

    target_file = csv_files[0]
    print(f"📦 Found and loading dataset: '{target_file}'")
    df = pd.read_csv(target_file)

    print(f"✅ Ingestion successful. Matrix footprint: {df.shape[0]:,} transactions across {df.shape[1]} features.")

    # 1. Dynamic Target Mapping Strategy
    target_candidates = ['Class', 'isFraud', 'is_fraud', 'fraud', 'Label']
    target_col = None
    for candidate in target_candidates:
        if candidate in df.columns:
            target_col = candidate
            break

    if not target_col:
        raise KeyError(f"❌ Could not auto-detect target class. Standard columns searched: {target_candidates}")

    print(f"🎯 Auto-detected target response column: '{target_col}'")

    # 2. Baseline Financial Damage Summary
    total_tx = len(df)
    fraud_cases = df[df[target_col] == 1]
    clean_cases = df[df[target_col] == 0]
    fraud_count = len(fraud_cases)
    fraud_rate = fraud_count / total_tx

    print("\n" + "="*50)
    print("      📈 EXECUTIVE CAPITAL PROFILE (BASELINE)      ")
    print("="*50)
    print(f"Total Processed Transactions: {total_tx:,}")
    print(f"Legitimate Transactions:     {len(clean_cases):,} ({1-fraud_rate:.4%})")
    print(f"Fraudulent Transactions:     {fraud_count:,} ({fraud_rate:.4%})")

    # Look for an transaction amount column to run financial impact math
    amt_col = [c for c in df.columns if c.lower() in ['amount', 'tx_amount', 'tx_amt']]
    has_amount = len(amt_col) > 0

    if has_amount:
        actual_amt_col = amt_col[0]
        total_fraud_loss = fraud_cases[actual_amt_col].sum()
        avg_fraud_loss = fraud_cases[actual_amt_col].mean()
        avg_clean_tx = clean_cases[actual_amt_col].mean()
        print(f"Gross Capital Stolen (Fraud): ${total_fraud_loss:,.2f}")
        print(f"Average Fraud Value:          ${avg_fraud_loss:,.2f} vs Clean Avg: ${avg_clean_tx:,.2f}")
    print("="*50)

    # 3. Features Pipeline Preparation
    # Drop IDs, strings, or targets to focus purely on mathematical signals
    exclude_cols = [target_col] + [c for c in df.columns if df[c].dtype == 'object' or 'id' in c.lower() or 'name' in c.lower()]
    X = df.drop(columns=exclude_cols)
    y = df[target_col]

    # Split using stratification to keep the needle in both the training and testing haystacks!
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

    # 4. Train Imbalance-Aware Classifier
    print("\n⚡ Training Risk Engine Classifier (Using Class-Weighted Balanced Learning Forest)...")
    model = RandomForestClassifier(
        n_estimators=50,
        max_depth=12,
        class_weight='balanced', # Crucial: tells the model that catching 1 fraud is worth way more than 1 clean transaction
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    # 5. Evaluate Operational Metrics
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    cm = confusion_matrix(y_test, y_pred)
    tp = cm[1, 1]  # Caught fraud
    fn = cm[1, 0]  # Missed fraud (Slipped through)
    fp = cm[0, 1]  # False Alarm (Legitimate user blocked)
    tn = cm[0, 0]  # Clean transaction approved safely

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print("\n" + "="*50)
    print("     🛡️ BACKEND FRAUD ENGINE AUDIT LOG (TEST SPLIT)     ")
    print("="*50)
    print(f"✅ Legitimate Approvals (True Negatives):  {tn:,}")
    print(f"🚨 Successfully Deflected Fraud (True Positives): {tp:,}")
    print(f"🛑 False Alarms Triggered (False Positives):    {fp:,}")
    print(f"💸 Slipped Fraud Incidents (False Negatives):   {fn:,}")
    print("-"*50)
    print(f"Precision (When model flags fraud, how often is it right?): {precision:.2%}")
    print(f"Recall (What % of total actual fraud did we catch?):        {recall:.2%}")
    print(f"Engine F1-Score Baseline:                                  {f1:.4f}")

    # 6. Cost-Benefit Simulation Matrix
    # Business assumptions: False alarm cost = $5 (support call time). Missed fraud cost = actual transaction cost (or generic $150 average if amount column is missing)
    cost_per_fp = 5.0
    if has_amount:
        cost_per_fn = df.loc[y_test[y_test == 1].index, actual_amt_col].mean()
    else:
        cost_per_fn = 150.0 # Default fallback financial weight

    baseline_unprotected_loss = (tp + fn) * cost_per_fn
    engine_operating_loss = (fp * cost_per_fp) + (fn * cost_per_fn)
    net_savings = baseline_unprotected_loss - engine_operating_loss

    print("\n" + "="*50)
    print("     💵 EXECUTIVE FINANCIAL MITIGATION METRICS     ")
    print("="*50)
    print(f"Unprotected Potential Damage:         ${baseline_unprotected_loss:,.2f}")
    print(f"Operating Cost with Engine Active:     ${engine_operating_loss:,.2f}")
    print(f"💰 NET CAPITAL PRESERVED BY INSTANCE:  ${net_savings:,.2f}")
    print("="*50)

    # 7. Extract Top 3 Risk Signatures
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    print("\n🔍 TOP RISK SIGNATURES IDENTIFIED BY ENGINE:")
    for i in range(min(3, len(X.columns))):
        print(f"  {i+1}. Feature '{X.columns[indices[i]]}' (Signal Strength: {importances[indices[i]]:.2%})")

if __name__ == "__main__":
    run_fraud_risk_engine()

🚀 Ingesting heavy-duty transactional financial data...
📦 Found and loading dataset: 'test.csv'
✅ Ingestion successful. Matrix footprint: 174,681 transactions across 23 features.
🎯 Auto-detected target response column: 'is_fraud'

      📈 EXECUTIVE CAPITAL PROFILE (BASELINE)      
Total Processed Transactions: 174,681
Legitimate Transactions:     173,938 (99.5752%)
Fraudulent Transactions:     742 (0.4248%)


ValueError: Input y contains NaN.

In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc

def run_fraud_risk_engine():
    print("🚀 Ingesting heavy-duty transactional financial data...")

    # Locate the file dynamically in the workspace
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError("❌ No CSV file found. Please upload your Kaggle dataset into the Colab side-panel!")

    target_file = csv_files[0]
    print(f"📦 Found and loading dataset: '{target_file}'")
    df = pd.read_csv(target_file)

    print(f"✅ Ingestion successful. Matrix footprint: {df.shape[0]:,} transactions across {df.shape[1]} features.")

    # 1. Dynamic Target Mapping Strategy
    target_candidates = ['Class', 'isFraud', 'is_fraud', 'fraud', 'Label']
    target_col = None
    for candidate in target_candidates:
        if candidate in df.columns:
            target_col = candidate
            break

    if not target_col:
        raise KeyError(f"❌ Could not auto-detect target class. Standard columns searched: {target_candidates}")

    print(f"🎯 Auto-detected target response column: '{target_col}'")

    # 2. Baseline Financial Damage Summary
    total_tx = len(df)
    fraud_cases = df[df[target_col] == 1]
    clean_cases = df[df[target_col] == 0]
    fraud_count = len(fraud_cases)
    fraud_rate = fraud_count / total_tx

    print("\n" + "="*50)
    print("      📈 EXECUTIVE CAPITAL PROFILE (BASELINE)      ")
    print("="*50)
    print(f"Total Processed Transactions: {total_tx:,}")
    print(f"Legitimate Transactions:     {len(clean_cases):,} ({1-fraud_rate:.4%})")
    print(f"Fraudulent Transactions:     {fraud_count:,} ({fraud_rate:.4%})")

    # Look for an transaction amount column to run financial impact math
    amt_col = [c for c in df.columns if c.lower() in ['amount', 'tx_amount', 'tx_amt']]
    has_amount = len(amt_col) > 0

    if has_amount:
        actual_amt_col = amt_col[0]
        total_fraud_loss = fraud_cases[actual_amt_col].sum()
        avg_fraud_loss = fraud_cases[actual_amt_col].mean()
        avg_clean_tx = clean_cases[actual_amt_col].mean()
        print(f"Gross Capital Stolen (Fraud): ${total_fraud_loss:,.2f}")
        print(f"Average Fraud Value:          ${avg_fraud_loss:,.2f} vs Clean Avg: ${avg_clean_tx:,.2f}")
    print("="*50)

    # 3. Features Pipeline Preparation
    # Drop IDs, strings, or targets to focus purely on mathematical signals
    exclude_cols = [target_col] + [c for c in df.columns if df[c].dtype == 'object' or 'id' in c.lower() or 'name' in c.lower()]
    X = df.drop(columns=exclude_cols)
    y = df[target_col]

    # Split using stratification to keep the needle in both the training and testing haystacks!
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

    # 4. Train Imbalance-Aware Classifier
    print("\n⚡ Training Risk Engine Classifier (Using Class-Weighted Balanced Learning Forest)...")
    model = RandomForestClassifier(
        n_estimators=50,
        max_depth=12,
        class_weight='balanced', # Crucial: tells the model that catching 1 fraud is worth way more than 1 clean transaction
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    # 5. Evaluate Operational Metrics
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    cm = confusion_matrix(y_test, y_pred)
    tp = cm[1, 1]  # Caught fraud
    fn = cm[1, 0]  # Missed fraud (Slipped through)
    fp = cm[0, 1]  # False Alarm (Legitimate user blocked)
    tn = cm[0, 0]  # Clean transaction approved safely

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print("\n" + "="*50)
    print("     🛡️ BACKEND FRAUD ENGINE AUDIT LOG (TEST SPLIT)     ")
    print("="*50)
    print(f"✅ Legitimate Approvals (True Negatives):  {tn:,}")
    print(f"🚨 Successfully Deflected Fraud (True Positives): {tp:,}")
    print(f"🛑 False Alarms Triggered (False Positives):    {fp:,}")
    print(f"💸 Slipped Fraud Incidents (False Negatives):   {fn:,}")
    print("-"*50)
    print(f"Precision (When model flags fraud, how often is it right?): {precision:.2%}")
    print(f"Recall (What % of total actual fraud did we catch?):        {recall:.2%}")
    print(f"Engine F1-Score Baseline:                                  {f1:.4f}")

    # 6. Cost-Benefit Simulation Matrix
    # Business assumptions: False alarm cost = $5 (support call time). Missed fraud cost = actual transaction cost (or generic $150 average if amount column is missing)
    cost_per_fp = 5.0
    if has_amount:
        cost_per_fn = df.loc[y_test[y_test == 1].index, actual_amt_col].mean()
    else:
        cost_per_fn = 150.0 # Default fallback financial weight

    baseline_unprotected_loss = (tp + fn) * cost_per_fn
    engine_operating_loss = (fp * cost_per_fp) + (fn * cost_per_fn)
    net_savings = baseline_unprotected_loss - engine_operating_loss

    print("\n" + "="*50)
    print("     💵 EXECUTIVE FINANCIAL MITIGATION METRICS     ")
    print("="*50)
    print(f"Unprotected Potential Damage:         ${baseline_unprotected_loss:,.2f}")
    print(f"Operating Cost with Engine Active:     ${engine_operating_loss:,.2f}")
    print(f"💰 NET CAPITAL PRESERVED BY INSTANCE:  ${net_savings:,.2f}")
    print("="*50)

    # 7. Extract Top 3 Risk Signatures
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    print("\n🔍 TOP RISK SIGNATURES IDENTIFIED BY ENGINE:")
    for i in range(min(3, len(X.columns))):
        print(f"  {i+1}. Feature '{X.columns[indices[i]]}' (Signal Strength: {importances[indices[i]]:.2%})")

if __name__ == "__main__":
    run_fraud_risk_engine()

🚀 Ingesting heavy-duty transactional financial data...
📦 Found and loading dataset: 'train.csv'
✅ Ingestion successful. Matrix footprint: 11,706 transactions across 23 features.
🎯 Auto-detected target response column: 'is_fraud'

      📈 EXECUTIVE CAPITAL PROFILE (BASELINE)      
Total Processed Transactions: 11,706
Legitimate Transactions:     11,657 (99.5900%)
Fraudulent Transactions:     48 (0.4100%)


ValueError: Input y contains NaN.

In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

def run_fraud_risk_engine():
    print("🚀 Ingesting heavy-duty transactional financial data...")

    # Locate the file dynamically in the workspace
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError("❌ No CSV file found. Please upload your Kaggle dataset into the Colab side-panel!")

    target_file = csv_files[0]
    print(f"📦 Found and loading dataset: '{target_file}'")
    df = pd.read_csv(target_file)

    print(f"✅ Ingestion successful. Matrix footprint: {df.shape[0]:,} transactions across {df.shape[1]} features.")

    # 1. Dynamic Target Mapping Strategy
    target_candidates = ['Class', 'isFraud', 'is_fraud', 'fraud', 'Label']
    target_col = None
    for candidate in target_candidates:
        if candidate in df.columns:
            target_col = candidate
            break

    if not target_col:
        raise KeyError(f"❌ Could not auto-detect target class. Standard columns searched: {target_candidates}")

    print(f"🎯 Auto-detected target response column: '{target_col}'")

    # 🔥 CRITICAL FIX: Clean out any rows where the target label is NaN/Blank
    initial_count = len(df)
    df = df.dropna(subset=[target_col])
    dropped_count = initial_count - len(df)
    if dropped_count > 0:
        print(f"🧹 Cleaned {dropped_count} row(s) with missing target labels.")

    # 2. Baseline Financial Damage Summary
    total_tx = len(df)
    fraud_cases = df[df[target_col] == 1]
    clean_cases = df[df[target_col] == 0]
    fraud_count = len(fraud_cases)
    fraud_rate = fraud_count / total_tx

    print("\n" + "="*50)
    print("      📈 EXECUTIVE CAPITAL PROFILE (BASELINE)      ")
    print("="*50)
    print(f"Total Processed Transactions: {total_tx:,}")
    print(f"Legitimate Transactions:     {len(clean_cases):,} ({1-fraud_rate:.4%})")
    print(f"Fraudulent Transactions:     {fraud_count:,} ({fraud_rate:.4%})")

    # Look for an transaction amount column to run financial impact math
    amt_col = [c for c in df.columns if c.lower() in ['amount', 'tx_amount', 'tx_amt', 'price']]
    has_amount = len(amt_col) > 0

    if has_amount:
        actual_amt_col = amt_col[0]
        total_fraud_loss = fraud_cases[actual_amt_col].sum()
        avg_fraud_loss = fraud_cases[actual_amt_col].mean()
        avg_clean_tx = clean_cases[actual_amt_col].mean()
        print(f"Gross Capital Stolen (Fraud): ${total_fraud_loss:,.2f}")
        print(f"Average Fraud Value:          ${avg_fraud_loss:,.2f} vs Clean Avg: ${avg_clean_tx:,.2f}")
    print("="*50)

    # 3. Features Pipeline Preparation
    # Drop IDs, strings, or targets to focus purely on mathematical signals
    exclude_cols = [target_col] + [c for c in df.columns if df[c].dtype == 'object' or 'id' in c.lower() or 'name' in c.lower()]
    X = df.drop(columns=exclude_cols)

    # 🔥 CRITICAL FIX 2: Impute any missing values in the feature matrix with column medians
    X = X.fillna(X.median())
    y = df[target_col].astype(int)

    # Split using stratification to keep the needle in both the training and testing haystacks!
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

    # 4. Train Imbalance-Aware Classifier
    print("\n⚡ Training Risk Engine Classifier (Using Class-Weighted Balanced Learning Forest)...")
    model = RandomForestClassifier(
        n_estimators=50,
        max_depth=12,
        class_weight='balanced', # Crucial: tells the model that catching 1 fraud is worth way more than 1 clean transaction
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    # 5. Evaluate Operational Metrics
    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    tp = cm[1, 1]  # Caught fraud
    fn = cm[1, 0]  # Missed fraud (Slipped through)
    fp = cm[0, 1]  # False Alarm (Legitimate user blocked)
    tn = cm[0, 0]  # Clean transaction approved safely

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print("\n" + "="*50)
    print("     🛡️ BACKEND FRAUD ENGINE AUDIT LOG (TEST SPLIT)     ")
    print("="*50)
    print(f"✅ Legitimate Approvals (True Negatives):  {tn:,}")
    print(f"🚨 Successfully Deflected Fraud (True Positives): {tp:,}")
    print(f"🛑 False Alarms Triggered (False Positives):    {fp:,}")
    print(f"💸 Slipped Fraud Incidents (False Negatives):   {fn:,}")
    print("-"*50)
    print(f"Precision (When model flags fraud, how often is it right?): {precision:.2%}")
    print(f"Recall (What % of total actual fraud did we catch?):        {recall:.2%}")
    print(f"Engine F1-Score Baseline:                                  {f1:.4f}")

    # 6. Cost-Benefit Simulation Matrix
    cost_per_fp = 5.0
    if has_amount:
        cost_per_fn = df.loc[y_test[y_test == 1].index, actual_amt_col].mean()
    else:
        cost_per_fn = 150.0 # Default fallback financial weight

    baseline_unprotected_loss = (tp + fn) * cost_per_fn
    engine_operating_loss = (fp * cost_per_fp) + (fn * cost_per_fn)
    net_savings = baseline_unprotected_loss - engine_operating_loss

    print("\n" + "="*50)
    print("     💵 EXECUTIVE FINANCIAL MITIGATION METRICS     ")
    print("="*50)
    print(f"Unprotected Potential Damage:         ${baseline_unprotected_loss:,.2f}")
    print(f"Operating Cost with Engine Active:     ${engine_operating_loss:,.2f}")
    print(f"💰 NET CAPITAL PRESERVED BY INSTANCE:  ${net_savings:,.2f}")
    print("="*50)

    # 7. Extract Top 3 Risk Signatures
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    print("\n🔍 TOP RISK SIGNATURES IDENTIFIED BY ENGINE:")
    for i in range(min(3, len(X.columns))):
        print(f"  {i+1}. Feature '{X.columns[indices[i]]}' (Signal Strength: {importances[indices[i]]:.2%})")

if __name__ == "__main__":
    run_fraud_risk_engine()

🚀 Ingesting heavy-duty transactional financial data...
📦 Found and loading dataset: 'train.csv'
✅ Ingestion successful. Matrix footprint: 1,296,675 transactions across 23 features.
🎯 Auto-detected target response column: 'is_fraud'

      📈 EXECUTIVE CAPITAL PROFILE (BASELINE)      
Total Processed Transactions: 1,296,675
Legitimate Transactions:     1,289,169 (99.4211%)
Fraudulent Transactions:     7,506 (0.5789%)

⚡ Training Risk Engine Classifier (Using Class-Weighted Balanced Learning Forest)...

     🛡️ BACKEND FRAUD ENGINE AUDIT LOG (TEST SPLIT)     
✅ Legitimate Approvals (True Negatives):  308,816
🚨 Successfully Deflected Fraud (True Positives): 1,579
🛑 False Alarms Triggered (False Positives):    13,476
💸 Slipped Fraud Incidents (False Negatives):   298
--------------------------------------------------
Precision (When model flags fraud, how often is it right?): 10.49%
Recall (What % of total actual fraud did we catch?):        84.12%
Engine F1-Score Baseline:                 